# MLflow Integration - AutoML Mock Data with OpenShift Authentication

This notebook demonstrates MLflow SDK integration for AutoML, following the same pattern as the working demo.

**Prerequisites:**
```bash
oc login -u <username> -p '<password>' https://api.ai-eng-gpu.socc.p3.openshiftapps.com:443
export MLFLOW_WORKSPACE="ai-eng-cracow"
export MLFLOW_TRACKING_URI="https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow"
export MLFLOW_TRACKING_TOKEN="$(oc whoami --show-token)"
```

Reference: `/architecture-decision-records/documentation/components/automl/features/mlflow_integration.md`

## Imports and Setup

In [7]:
import os
import json
import subprocess
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

print(f"MLflow version: {mlflow.__version__}")

MLflow version: 3.12.0


## Configure MLflow (Following Working Demo Pattern)

Set environment variables and configure MLflow exactly like the working demo script.

In [8]:
# Step 1: Get authentication token (if not already set)
if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
    try:
        token = subprocess.run(
            ["oc", "whoami", "--show-token"],
            capture_output=True,
            text=True,
            timeout=5,
            check=True
        ).stdout.strip()
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print("✅ Token retrieved from 'oc whoami --show-token'")
    except Exception as e:
        print(f"⚠️  Could not get token from oc CLI: {e}")
        print("   Set MLFLOW_TRACKING_TOKEN manually if needed")

# Step 2: Set MLflow environment variables (keep them in environment!)
MLFLOW_WORKSPACE = os.environ.get("MLFLOW_WORKSPACE", "ai-eng-cracow")
MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI",
    "https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow"
)

# Ensure environment variables are set (following working demo pattern)
os.environ["MLFLOW_WORKSPACE"] = MLFLOW_WORKSPACE
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI

print(f"\nMLflow Configuration:")
print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"  Workspace: {MLFLOW_WORKSPACE}")
print(f"  Token: {'✅ Set' if os.environ.get('MLFLOW_TRACKING_TOKEN') else '❌ Not set'}")


MLflow Configuration:
  Tracking URI: https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow
  Workspace: ai-eng-cracow
  Token: ✅ Set


## Set MLflow Experiment

Use `mlflow.set_experiment()` like the working demo (it handles everything automatically).

In [9]:
# Set experiment (following working demo pattern)
EXPERIMENT_NAME = "AutoML-Tabular-XXX-Training"

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    print(f"✅ Experiment set: {EXPERIMENT_NAME}")
    
    # Get experiment info
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment:
        print(f"   Experiment ID: {experiment.experiment_id}")
        print(f"   Artifact Location: {experiment.artifact_location}")
except Exception as e:
    print(f"❌ Error setting experiment: {e}")
    raise

✅ Experiment set: AutoML-Tabular-XXX-Training
   Experiment ID: 6
   Artifact Location: /mlflow/artifacts/workspaces/ai-eng-cracow/6


## Mock AutoML Data

In [10]:
# KFP Pipeline Context
kfp_context = {
    "pipeline_name": "autogluon-tabular-training-pipeline",
    "run_id": "run-abc123-def456-789ghi",
    "run_name": "autogluon-training-2026-05-07-143022",
    "kfp_version": "2.8.0"
}

# Pipeline Parameters
pipeline_params = {
    "task_type": "multiclass",
    "eval_metric": "accuracy",
    "preset": "medium_quality",
    "top_n": 3
}

# Mock AutoGluon metrics.json
all_metrics = {
    "WeightedEnsemble_L3": {
        "score_val": 0.9847,
        "fit_time": 145.23,
        "pred_time_val": 0.52,
        "accuracy": 0.9847,
        "f1": 0.9835,
        "roc_auc": 0.9962
    },
    "LightGBM_L2": {
        "score_val": 0.9782,
        "fit_time": 89.45,
        "pred_time_val": 0.34,
        "accuracy": 0.9782,
        "f1": 0.9771,
        "roc_auc": 0.9934
    },
    "RandomForest_L2": {
        "score_val": 0.9721,
        "fit_time": 112.67,
        "pred_time_val": 0.41,
        "accuracy": 0.9721,
        "f1": 0.9708,
        "roc_auc": 0.9911
    },
    "XGBoost_L2": {
        "score_val": 0.9756,
        "fit_time": 95.12,
        "pred_time_val": 0.38,
        "accuracy": 0.9756,
        "f1": 0.9744,
        "roc_auc": 0.9928
    },
    "CatBoost_L1": {
        "score_val": 0.9634,
        "fit_time": 156.89,
        "pred_time_val": 0.29,
        "accuracy": 0.9634,
        "f1": 0.9618,
        "roc_auc": 0.9881
    },
    "NeuralNetFastAI_L1": {
        "score_val": 0.9567,
        "fit_time": 203.45,
        "pred_time_val": 0.67,
        "accuracy": 0.9567,
        "f1": 0.9548,
        "roc_auc": 0.9845
    }
}

model_names = list(all_metrics.keys())
models_artifact_path = "/mnt/models/autogluon-tabular-2026-05-07"

print(f"✅ Mock data prepared for {len(model_names)} models")

✅ Mock data prepared for 6 models


## Log AutoML Results to MLflow

In [11]:
def log_automl_to_mlflow(
    all_metrics: dict,
    model_names: list,
    pipeline_params: dict,
    kfp_context: dict,
    models_artifact_path: str,
    workspace: str,
    log_artifacts: bool = False
):
    """
    Log AutoML results to MLflow using PARENT-CHILD RUN STRUCTURE.
    
    Following industry standard pattern (Optuna, FLAML, PySpark):
    - Parent run = KFP pipeline execution
    - Child runs = individual trained models (nested)
    
    Aligned with: https://github.com/LukaszCmielowski/architecture-decision-records/blob/autox_docs_updates/documentation/components/automl/features/mlflow_integration.md
    
    Args:
        log_artifacts: If True, attempts to log artifact files. 
                      Set to False when using remote MLflow with local artifact storage.
    """
    
    print("\n" + "="*60)
    print("LOGGING AUTOML TO MLFLOW - PARENT-CHILD PATTERN")
    print(f"Pipeline: {kfp_context['run_name']}")
    print(f"Models: {len(model_names)}")
    print("="*60)
    
    # ================================================================
    # PARENT RUN: Represents the KFP pipeline execution
    # ================================================================
    with mlflow.start_run(run_name=kfp_context['run_name']) as parent_run:
        
        parent_run_id = parent_run.info.run_id
        print(f"\n✅ Parent run created: {parent_run_id}")
        print(f"   Represents: KFP pipeline execution")
        
        # ============================================================
        # PARENT: Tags (per updated design doc)
        # ============================================================
        print("\n📌 Logging pipeline-level TAGS to parent...")
        parent_tags = {
            "pipeline_name": kfp_context['pipeline_name'],
            "kfp_run_name": kfp_context['run_name'],
            "kfp_run_id": kfp_context['run_id'],
            "kfp_version": kfp_context['kfp_version'],
            "image": "quay.io/opendatahub/automl:v1.2.3",
            "autogluon_version": "1.1.1"
        }
        mlflow.set_tags(parent_tags)
        
        # ============================================================
        # PARENT: Parameters (per updated design doc)
        # ============================================================
        print("⚙️  Logging pipeline-level PARAMS to parent...")
        
        # Calculate aggregate statistics
        scores = [m['score_val'] for m in all_metrics.values()]
        fit_times = [m['fit_time'] for m in all_metrics.values()]
        
        parent_params = {
            "task_type": pipeline_params['task_type'],
            "eval_metric": pipeline_params['eval_metric'],
            "preset": pipeline_params['preset'],
            "top_n": str(pipeline_params['top_n']),
            "dataset_hash": "sha256:1a2b3c4d5e6f7g8h9i0j",
            "dataset_uri": "s3://my-bucket/datasets/iris.csv"
        }
        mlflow.log_params(parent_params)
        
        # ============================================================
        # PARENT: Metrics (aggregate statistics per design doc)
        # ============================================================
        print("📊 Logging pipeline-level METRICS to parent...")
        
        # Per design doc: best_score, worst_score, mean_score, num_models_trained, total_fit_time_seconds
        parent_metrics = {
            "best_score": max(scores),
            "worst_score": min(scores),
            "mean_score": sum(scores) / len(scores),
            "num_models_trained": len(model_names),
            "total_fit_time_seconds": sum(fit_times)
        }
        mlflow.log_metrics(parent_metrics)
        
        print(f"   - best_score: {parent_metrics['best_score']:.4f}")
        print(f"   - worst_score: {parent_metrics['worst_score']:.4f}")
        print(f"   - mean_score: {parent_metrics['mean_score']:.4f}")
        print(f"   - num_models_trained: {parent_metrics['num_models_trained']}")
        print(f"   - total_fit_time_seconds: {parent_metrics['total_fit_time_seconds']:.2f}s")
        
        # ============================================================
        # PARENT: Artifacts (shared pipeline artifacts)
        # ============================================================
        if log_artifacts:
            try:
                print("📄 Logging pipeline-level ARTIFACTS to parent...")
                
                # Create leaderboard HTML
                leaderboard_html = """<html>
<head><title>AutoGluon Leaderboard</title></head>
<body>
    <h1>AutoGluon Leaderboard</h1>
    <table border="1">
        <tr><th>Rank</th><th>Model</th><th>Score</th><th>Fit Time</th><th>Predict Time</th></tr>
"""
                for rank, name in enumerate(sorted(all_metrics.keys(), 
                                                   key=lambda x: all_metrics[x]['score_val'], 
                                                   reverse=True), 1):
                    m = all_metrics[name]
                    leaderboard_html += f"""        <tr>
            <td>{rank}</td>
            <td>{name}</td>
            <td>{m['score_val']:.4f}</td>
            <td>{m['fit_time']:.2f}s</td>
            <td>{m['pred_time_val']:.2f}s</td>
        </tr>
"""
                leaderboard_html += """    </table>
</body>
</html>"""
                
                leaderboard_path = Path("leaderboard.html")
                leaderboard_path.write_text(leaderboard_html)
                mlflow.log_artifact(str(leaderboard_path), artifact_path="reports")
                leaderboard_path.unlink()
                print("   - leaderboard.html")
            except Exception as e:
                print(f"    ⚠️  Could not log artifacts: {e}")
        
        # ============================================================
        # CHILD RUNS: One nested run per model
        # ============================================================
        print(f"\n👶 Creating {len(model_names)} CHILD RUNS (nested)...")
        print("─"*60)
        
        child_run_ids = []
        
        for i, model_name in enumerate(model_names, 1):
            model_metrics = all_metrics[model_name]
            
            # Parse model info
            model_type = model_name.split("_")[0] if "_" in model_name else model_name
            stack_level = int(model_name.split("_L")[-1]) if "_L" in model_name else 1
            
            # Create nested child run for this model
            with mlflow.start_run(run_name=model_name, nested=True) as child_run:
                
                child_run_id = child_run.info.run_id
                child_run_ids.append(child_run_id)
                
                # --------------------------------------------------------
                # CHILD: Tags (model identity)
                # --------------------------------------------------------
                mlflow.set_tags({
                    "model_name": model_name,
                    "model_type": model_type,
                    "stack_level": str(stack_level)
                })
                
                # --------------------------------------------------------
                # CHILD: Parameters (model-specific)
                # --------------------------------------------------------
                mlflow.log_params({
                    "model_type": model_type,
                    "stack_level": stack_level,
                    "fit_time": model_metrics['fit_time'],
                    "predict_time": model_metrics['pred_time_val'],
                    "num_bag_folds": 8 if stack_level > 1 else 0,
                    "num_stack_levels": 3,
                    # Artifact pointers (paths to KFP artifacts)
                    "metrics_path": f"{models_artifact_path}/{model_name}_FULL/metrics",
                    "predictor_path": f"{models_artifact_path}/{model_name}_FULL/predictor",
                    "notebook_path": f"{models_artifact_path}/{model_name}_FULL/notebooks/automl_predictor_notebook.ipynb"
                })
                
                # --------------------------------------------------------
                # CHILD: Metrics (model performance)
                # --------------------------------------------------------
                metrics_to_log = {"score_val": model_metrics['score_val']}
                for metric in ['accuracy', 'f1', 'roc_auc', 'rmse', 'mae']:
                    if metric in model_metrics:
                        metrics_to_log[metric] = model_metrics[metric]
                mlflow.log_metrics(metrics_to_log)
                
                print(f"   [{i}/{len(model_names)}] {model_name:<25} | Score: {model_metrics['score_val']:.4f} | ID: {child_run_id[:8]}...")
        
        print("\n" + "="*60)
        print("✅ LOGGING COMPLETED")
        print("="*60)
        print(f"Parent Run ID: {parent_run_id}")
        print(f"Child Runs: {len(child_run_ids)}")
        print(f"Best Score: {parent_metrics['best_score']:.4f}")
        print(f"Total Training Time: {parent_metrics['total_fit_time_seconds']:.2f}s")
        print(f"\n💡 Parent-child structure follows industry standard (Optuna/FLAML)")
        
        # Return tracking info
        experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        return {
            "tracking_enabled": True,
            "mlflow_tracking_uri": mlflow.get_tracking_uri(),
            "mlflow_experiment_id": experiment.experiment_id,
            "mlflow_experiment_name": experiment.name,
            "mlflow_parent_run_id": parent_run_id,
            "mlflow_child_run_ids": child_run_ids,
            "mlflow_workspace": workspace,
            "total_models": len(child_run_ids),
            "best_score": parent_metrics['best_score'],
            "worst_score": parent_metrics['worst_score'],
            "mean_score": parent_metrics['mean_score'],
            "total_fit_time_seconds": parent_metrics['total_fit_time_seconds'],
            "kfp_run_id": kfp_context['run_id'],
            "pipeline_name": kfp_context['pipeline_name'],
            "model_names": model_names
        }

# Execute logging with parent-child structure
tracking_info = log_automl_to_mlflow(
    all_metrics=all_metrics,
    model_names=model_names,
    pipeline_params=pipeline_params,
    kfp_context=kfp_context,
    models_artifact_path=models_artifact_path,
    workspace=MLFLOW_WORKSPACE,
    log_artifacts=False  # Set to True only if artifact storage is writable
)

🏃 View run autogluon-training-2026-05-07-143022 at: https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow/#/experiments/6/runs/2c57f9e56a954e4183ae4d2545dc9509?workspace=ai-eng-cracow
🧪 View experiment at: https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow/#/experiments/6?workspace=ai-eng-cracow


## Query Logged Data

In [12]:
client = MlflowClient()

print("="*60)
print("QUERYING PARENT-CHILD RUN STRUCTURE")
print("="*60)

parent_run_id = tracking_info['mlflow_parent_run_id']
experiment_id = tracking_info['mlflow_experiment_id']

# ============================================================
# Query Parent Run
# ============================================================
parent_run = client.get_run(parent_run_id)

print(f"\n🎯 PARENT RUN (Pipeline Execution):")
print(f"{'─'*60}")
print(f"Run ID: {parent_run.info.run_id}")
print(f"Run Name: {parent_run.info.run_name}")
print(f"Status: {parent_run.info.status}")

print(f"\n  📌 Tags:")
for k, v in sorted(parent_run.data.tags.items()):
    if not k.startswith('mlflow.'):
        print(f"    {k}: {v}")

print(f"\n  ⚙️  Parameters:")
for k, v in sorted(parent_run.data.params.items()):
    print(f"    {k}: {v}")

print(f"\n  📊 Metrics:")
for k, v in sorted(parent_run.data.metrics.items()):
    print(f"    {k}: {v}")

# ============================================================
# Query Child Runs
# ============================================================
child_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'"
)

print(f"\n👶 CHILD RUNS ({len(child_runs)} models):")
print(f"{'─'*60}")

# Sort by score descending
child_runs_sorted = sorted(child_runs, key=lambda r: r.data.metrics.get('score_val', 0), reverse=True)

for i, child in enumerate(child_runs_sorted, 1):
    model_name = child.data.tags.get('model_name', child.info.run_name)
    score = child.data.metrics.get('score_val', 0)
    model_type = child.data.params.get('model_type', 'N/A')
    stack_level = child.data.params.get('stack_level', 'N/A')
    
    print(f"\n  [{i}] {model_name}")
    print(f"      Run ID: {child.info.run_id[:16]}...")
    print(f"      Score: {score:.4f}")
    print(f"      Type: {model_type} | Stack Level: {stack_level}")
    print(f"      Metrics: ", end="")
    metrics = ['accuracy', 'f1', 'roc_auc']
    metric_strs = [f"{m}={child.data.metrics.get(m, 0):.4f}" for m in metrics if m in child.data.metrics]
    print(", ".join(metric_strs))

print(f"\n{'='*60}")
print(f"✅ Query completed - Parent-child hierarchy verified")
print(f"{'='*60}")
print(f"\n💡 Structure: 1 parent (pipeline) + {len(child_runs)} children (models)")


👶 CHILD RUNS (6 models):
────────────────────────────────────────────────────────────

  [1] WeightedEnsemble_L3
      Run ID: a51c31d8de0d4cb7...
      Score: 0.9847
      Type: WeightedEnsemble | Stack Level: 3
      Metrics: accuracy=0.9847, f1=0.9835, roc_auc=0.9962

  [2] LightGBM_L2
      Run ID: c23c79edd8884193...
      Score: 0.9782
      Type: LightGBM | Stack Level: 2
      Metrics: accuracy=0.9782, f1=0.9771, roc_auc=0.9934

  [3] XGBoost_L2
      Run ID: e1db21a697124279...
      Score: 0.9756
      Type: XGBoost | Stack Level: 2
      Metrics: accuracy=0.9756, f1=0.9744, roc_auc=0.9928

  [4] RandomForest_L2
      Run ID: 47e25430ed554d82...
      Score: 0.9721
      Type: RandomForest | Stack Level: 2
      Metrics: accuracy=0.9721, f1=0.9708, roc_auc=0.9911

  [5] CatBoost_L1
      Run ID: 627dc20648074fbd...
      Score: 0.9634
      Type: CatBoost | Stack Level: 1
      Metrics: accuracy=0.9634, f1=0.9618, roc_auc=0.9881

  [6] NeuralNetFastAI_L1
      Run ID: 96b1fb

---